# Lab 13 — Fine-Tuning with FINETUNE

Customize an LLM for your specific domain by training on your own examples.

| Item | Detail |
|---|---|
| **Function** | `SNOWFLAKE.CORTEX.FINETUNE('CREATE', ...)` |
| **Exam Domain** | 2.0 Gen AI & LLM Functions |
| **You'll learn** | Training data format, fine-tune creation, job monitoring, inference |


## Step 1 — Environment Setup

In [ ]:
USE ROLE DS_ROLE;
USE WAREHOUSE DS_WH;

CREATE DATABASE IF NOT EXISTS GENAI_LEARNING;
CREATE SCHEMA IF NOT EXISTS GENAI_LEARNING.PUBLIC;
USE DATABASE GENAI_LEARNING;
USE SCHEMA PUBLIC;

## Step 2 — Prepare Training Data

Fine-tuning requires a table with `prompt` and `completion` columns.
The model learns to produce `completion` given `prompt`.


In [ ]:
CREATE OR REPLACE TABLE finetune_training_data (
    prompt TEXT,
    completion TEXT
);

INSERT INTO finetune_training_data (prompt, completion) VALUES
('Classify this Snowflake query issue: Query took 45 minutes on XS warehouse with full table scan', 'Category: PERFORMANCE\nSeverity: HIGH\nRecommendation: Add clustering key or increase warehouse size. Consider pruning with WHERE clause.'),
('Classify this Snowflake query issue: User cannot see table in ANALYTICS schema', 'Category: ACCESS_CONTROL\nSeverity: MEDIUM\nRecommendation: Grant SELECT on schema ANALYTICS to user role. Check RBAC hierarchy.'),
('Classify this Snowflake query issue: COPY INTO command failing with file format error on CSV', 'Category: DATA_LOADING\nSeverity: MEDIUM\nRecommendation: Check FILE_FORMAT settings: field delimiter, skip_header, error_on_column_count_mismatch.'),
('Classify this Snowflake query issue: Credit consumption spiked 300% overnight', 'Category: COST\nSeverity: CRITICAL\nRecommendation: Check WAREHOUSE_METERING_HISTORY. Look for runaway queries or misconfigured auto-resume.'),
('Classify this Snowflake query issue: Streams showing stale data after table recreation', 'Category: DATA_PIPELINE\nSeverity: HIGH\nRecommendation: Recreate stream. Streams become stale when source table is recreated. Use ALTER TABLE instead.');

In [ ]:
SELECT * FROM finetune_training_data;

## Step 3 — Create a Fine-Tuning Job

```sql
-- Fine-tuning syntax (DO NOT RUN without sufficient credits):
SELECT SNOWFLAKE.CORTEX.FINETUNE(
    'CREATE',
    'snowflake_support_classifier',   -- job name
    'llama3.1-8b',                    -- base model
    'SELECT prompt, completion FROM finetune_training_data',
    ''                                 -- validation query (optional)
);
```

> **Note:** Fine-tuning consumes significant credits. This cell shows the syntax without executing.


## Step 4 — Monitor & Use Fine-Tuned Models

```sql
-- Check job status
SELECT SNOWFLAKE.CORTEX.FINETUNE('DESCRIBE', 'snowflake_support_classifier');

-- List all fine-tuning jobs
SELECT SNOWFLAKE.CORTEX.FINETUNE('SHOW');

-- Use the fine-tuned model (after completion)
SELECT SNOWFLAKE.CORTEX.COMPLETE(
    'snowflake_support_classifier',
    'Classify this Snowflake query issue: Dashboard refresh taking over 10 minutes'
);
```


## Exam-Relevant Concepts

| Concept | Detail |
|---|---|
| Base models for fine-tuning | `llama3.1-8b`, `mistral-7b` (smaller models only) |
| Training data format | Table with `prompt` and `completion` columns |
| Minimum training examples | ~100 recommended for meaningful improvement |
| Fine-tuned model usage | Same as any model in `COMPLETE()` |
| Job management | `CREATE`, `DESCRIBE`, `SHOW`, `CANCEL` operations |


## Key Takeaways

- Fine-tuning adapts a base model to your domain with `FINETUNE('CREATE', ...)`.
- Training data = table with `prompt` and `completion` columns.
- Use smaller base models (8b/7b) — not all models support fine-tuning.
- Monitor with `FINETUNE('DESCRIBE', ...)` and `FINETUNE('SHOW')`.
- Fine-tuned models are used in `COMPLETE()` just like built-in models.
